# Job Match Ranker — Multi-Model Comparison

## What is this notebook for?

In `02_train_model.ipynb` we trained LightGBM and it performed well. But how do we know it's actually the **best choice** and not just the first thing we tried? This notebook answers that — we train 5 different models on the exact same data and compare them side by side.

The goal: **prove that LightGBM is the right tool for this job**, or find out if something else beats it.

---

## Which 5 models are we comparing and why?

We picked models that are very different from each other — that way the comparison actually tells us something. If we just compared 5 similar models, we'd learn nothing useful.

| Model | Why we included it |
|---|---|
| **LightGBM** | This is already in production — it's the one we're trying to justify |
| **XGBoost** | The most famous tree-based model — the "obvious" alternative to LightGBM |
| **CatBoost** | A newer tree model that's often better when you have less data (we only have ~5k rows) |
| **Random Forest** | A completely different approach to trees — grows them all in parallel instead of one at a time |
| **Logistic Regression** | The simplest possible model — proves whether the complexity of the other models is actually worth it |

**Why no neural networks (deep learning)?**

Three practical reasons:
1. Neural networks can't handle missing values (`NaN`) directly. Our data has `NaN` for new freelancers with no history, and we'd need extra code to work around that.
2. They're too slow for a page load. The model runs every time a freelancer opens the homepage — we need results in under 20ms. Neural networks on a CPU are typically 10–50× slower than tree models for a 100-sample batch.
3. For structured table data with less than ~100k rows, tree models almost always beat deep learning. We only have ~5k rows.

**Why no collaborative filtering (like Netflix recommendations)?**

Collaborative filtering works by learning from past behaviour — "users who liked X also liked Y". We'd need a lot of completed proposals and contracts to build that kind of history. At launch, we don't have enough yet. Our 18 handcrafted features (skill overlap, rate fit, etc.) are a better fit for this early stage.

---

## What are we measuring?

| Metric | Plain English |
|---|---|
| **AUC-ROC** | Overall ranking quality — how well does the model separate good matches from bad ones? 1.0 = perfect, 0.5 = random guess |
| **Average Precision** | Similar to AUC-ROC but stricter when the data is imbalanced. Only ~6% of our pairs are actual good matches, so this matters |
| **Inference time** | How long does it take to score 100 jobs? Must stay under 20ms so the homepage doesn't feel slow |
| **Cold-start AUC** | AUC measured only on brand-new freelancers who have no history yet. A model that fails here breaks the experience for new users |
| **5-fold CV AUC** | We train and test 5 times on different data splits. If the score is stable across all 5, the model is solid — not just lucky on one split |

In [4]:
import numpy as np
import pandas as pd
import time
import json
import os
import warnings
warnings.filterwarnings('ignore')

!pip install catboost -qq
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, average_precision_score

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

np.random.seed(42)
DATA_DIR  = 'content'
MODEL_DIR = 'models'
os.makedirs(MODEL_DIR, exist_ok=True)

print('LightGBM:', lgb.__version__)
print('XGBoost: ', xgb.__version__)
print('CatBoost:', cb.__version__)
print('Setup complete')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.7 MB/s eta 0:00:00
LightGBM: 4.6.0
XGBoost:  3.2.0
CatBoost: 1.2.10
Setup complete


In [5]:
# ── Load data ─────────────────────────────────────────────────────────────────
ml_df = pd.read_csv(f'/{DATA_DIR}/ml_training_pairs.csv')
print(f'Loaded {len(ml_df):,} training pairs')
print(f'Label distribution:\n{ml_df["label"].value_counts()}')
print(f'Positive rate: {ml_df["label"].mean():.3f}')

FEATURE_COLS = [
    'cosine_sim',
    'skill_overlap_pct',
    'skill_required_matched',
    'skill_required_total',
    'skill_preferred_pct',
    'experience_level_match',
    'exp_delta',
    'rate_in_budget',
    'rate_ratio',
    'language_match',
    'speciality_match',
    'domain_match',
    'has_portfolio',
    'work_exp_count',
    'performance_score',
    'success_rate_hist',
    'total_projects',
    'is_cold_start',
]

X = ml_df[FEATURE_COLS].copy()
y = ml_df['label'].copy()

# ── Train / val / test split — same as 02_train_model.ipynb ──────────────────
X_tv, X_test, y_tv, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_tv, y_tv, test_size=0.15, stratify=y_tv, random_state=42)

print(f'\nTrain:      {len(X_train):,}  (pos={y_train.mean():.3f})')
print(f'Validation: {len(X_val):,}  (pos={y_val.mean():.3f})')
print(f'Test:       {len(X_test):,}  (pos={y_test.mean():.3f})')

Loaded 7,143 training pairs
Label distribution:
label
0    6720
1     423
Name: count, dtype: int64
Positive rate: 0.059

Train:      5,160  (pos=0.059)
Validation: 911  (pos=0.059)
Test:       1,072  (pos=0.059)


## Model 1 — LightGBM (our current production model)

This is the model already running in production. We include it here as the benchmark everyone else has to beat.

**How it works (simply):**

LightGBM builds decision trees one at a time, where each new tree tries to fix the mistakes the previous trees made. It's called "boosting" — like a team where each new member focuses on the cases the rest of the team got wrong.

What makes it special compared to the others:
- It grows trees **leaf by leaf**, always splitting the part of the tree that will reduce error the most. This makes it more aggressive and accurate than XGBoost's row-by-row approach.
- It **handles missing values natively** — during training it learns which way to route a `NaN` value at each split (whichever direction reduces error more). This is important for us because new freelancers have no performance history (`NaN`), and we don't want to break the model for them.
- It's **very fast at prediction time** because the tree structure is compiled to C++ — no Python code runs when scoring a job.

We expect this to be the winner or very close to it.

In [6]:
lgbm_model = lgb.LGBMClassifier(
    n_estimators=1000,
    learning_rate=0.025,
    max_depth=6,
    num_leaves=31,
    min_child_samples=20,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,
    class_weight='balanced',
    reg_alpha=0.1,
    reg_lambda=0.2,
    n_jobs=-1,
    random_state=42,
    verbose=-1,
)

lgbm_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[
        lgb.early_stopping(stopping_rounds=50, verbose=False),
        lgb.log_evaluation(period=200),
    ],
)

print(f'Best iteration: {lgbm_model.best_iteration_}')

lgbm_prob = lgbm_model.predict_proba(X_test)[:, 1]
lgbm_auc  = roc_auc_score(y_test, lgbm_prob)
lgbm_ap   = average_precision_score(y_test, lgbm_prob)
print(f'LightGBM  AUC-ROC: {lgbm_auc:.4f}  AP: {lgbm_ap:.4f}')

[200]	valid_0's binary_logloss: 0.180826
[400]	valid_0's binary_logloss: 0.166962
[600]	valid_0's binary_logloss: 0.162156
Best iteration: 617
LightGBM  AUC-ROC: 0.9679  AP: 0.5482


## Model 2 — XGBoost

XGBoost is probably the most well-known ML model in data science competitions. If you've heard of Kaggle, XGBoost wins a lot of those competitions. It's the natural "first challenger" to compare against LightGBM.

**How it differs from LightGBM:**

Both LightGBM and XGBoost build trees one at a time and correct mistakes from the previous tree (boosting). The main difference is *how* they grow each tree:

- **XGBoost** grows trees **level by level** — it finishes all the splits at depth 1, then all the splits at depth 2, etc. It's more careful and less likely to overfit on small datasets.
- **LightGBM** grows trees **leaf by leaf** — it always picks the single split that reduces error the most, regardless of which level it's at. This is more aggressive and usually finds better patterns faster.

XGBoost also handles `NaN` natively, so cold-start freelancers won't break it either.

**What we expect:** XGBoost should be close to LightGBM in accuracy, but probably a bit slower at prediction time because level-by-level growth produces slightly more complex trees.

In [7]:
# Compute scale_pos_weight to handle imbalance (XGBoost doesn't have class_weight='balanced')
neg_count  = int((y_train == 0).sum())
pos_count  = int((y_train == 1).sum())
spw        = neg_count / pos_count
print(f'scale_pos_weight = {spw:.1f}  (neg={neg_count}, pos={pos_count})')

xgb_model = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.025,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=0.2,
    scale_pos_weight=spw,          # handles class imbalance
    tree_method='hist',            # histogram-based (faster, matches LightGBM fairness)
    enable_categorical=False,
    eval_metric='logloss',
    early_stopping_rounds=50,
    n_jobs=-1,
    random_state=42,
    verbosity=0,
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=False,
)

print(f'Best iteration: {xgb_model.best_iteration}')

xgb_prob = xgb_model.predict_proba(X_test)[:, 1]
xgb_auc  = roc_auc_score(y_test, xgb_prob)
xgb_ap   = average_precision_score(y_test, xgb_prob)
print(f'XGBoost   AUC-ROC: {xgb_auc:.4f}  AP: {xgb_ap:.4f}')

scale_pos_weight = 15.9  (neg=4854, pos=306)
Best iteration: 413
XGBoost   AUC-ROC: 0.9681  AP: 0.5407


## Model 3 — CatBoost

CatBoost is another boosting model (same family as LightGBM and XGBoost), but it was built by Yandex (the Russian Google). It has a few tricks that make it interesting, especially for small datasets like ours.

**Why it might do well here:**

Our training set has ~5,000 rows — that's not a lot. With small data, there's a risk that the model learns the training data too well (overfitting) and performs worse on new data. CatBoost has two features designed to reduce this:

1. **Ordered boosting** — instead of training on the full dataset each time, it shuffles the data and trains on only part of it per round. This makes it harder for the model to "memorise" the training examples.
2. **Symmetric trees** — unlike LightGBM and XGBoost where each branch can split on a different feature, CatBoost uses the *same* split across an entire level of the tree. This acts as a built-in regulariser (it can't overfit as aggressively).

CatBoost also handles `NaN` values natively — it treats missing as its own separate category.

**What we expect:** CatBoost might actually beat LightGBM on this small dataset. Training will be slower, but prediction speed should be similar or faster due to the simpler symmetric trees.

In [8]:
cat_model = cb.CatBoostClassifier(
    iterations=1000,
    learning_rate=0.025,
    depth=6,
    subsample=0.8,
    colsample_bylevel=0.8,
    l2_leaf_reg=3.0,
    class_weights=[1.0, spw],      # same imbalance weight as XGBoost
    eval_metric='AUC',
    early_stopping_rounds=50,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
)

print(f'Best iteration: {cat_model.best_iteration_}')

cat_prob = cat_model.predict_proba(X_test)[:, 1]
cat_auc  = roc_auc_score(y_test, cat_prob)
cat_ap   = average_precision_score(y_test, cat_prob)
print(f'CatBoost  AUC-ROC: {cat_auc:.4f}  AP: {cat_ap:.4f}')

Best iteration: 64
CatBoost  AUC-ROC: 0.9673  AP: 0.5168


## Model 4 — Random Forest

Random Forest is the classic tree ensemble. It's fundamentally different from the three models above, which is exactly why we include it — it lets us test whether the boosting strategy is actually necessary.

**The key difference — bagging vs boosting:**

- **Boosting** (LightGBM, XGBoost, CatBoost): builds trees *one at a time*, each correcting the errors of the previous. The trees are dependent on each other.
- **Bagging** (Random Forest): builds *many independent trees in parallel* on random subsets of the data, then takes a vote. No tree knows about the others.

If Random Forest matches LightGBM in accuracy, it would mean the sequential error-correction of boosting isn't actually helping — we could use the simpler parallel approach instead.

**One important difference — NaN handling:**

Random Forest in scikit-learn *does not* handle `NaN` values natively. We have to fill in the missing values manually before training. We use `-1` as a placeholder — the trees will learn that `-1` in `performance_score` means "this freelancer has no history", especially since those rows also have `is_cold_start = 1`.

**What we expect:** Lower accuracy than LightGBM, and noticeably slower prediction because Random Forest runs 500 trees in parallel through Python rather than one optimised C++ pass.

In [9]:
# Random Forest doesn't handle NaN natively in sklearn — impute with sentinel -1
# The tree will learn that -1 in performance_score correlates with is_cold_start=1
X_train_rf = X_train.fillna(-1)
X_val_rf   = X_val.fillna(-1)
X_test_rf  = X_test.fillna(-1)

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=12,
    min_samples_leaf=10,
    max_features='sqrt',
    class_weight='balanced',
    n_jobs=-1,
    random_state=42,
)

rf_model.fit(X_train_rf, y_train)

rf_prob = rf_model.predict_proba(X_test_rf)[:, 1]
rf_auc  = roc_auc_score(y_test, rf_prob)
rf_ap   = average_precision_score(y_test, rf_prob)
print(f'Random Forest  AUC-ROC: {rf_auc:.4f}  AP: {rf_ap:.4f}')

Random Forest  AUC-ROC: 0.9687  AP: 0.5291


## Model 5 — Logistic Regression (the "dumb" baseline)

Logistic Regression is the simplest model we can reasonably use for binary classification. It's not actually "dumb" — it works great for many problems — but for this task it should be the weakest contender.

**How it works:**

It multiplies each feature by a learned weight, adds them all up, and converts the total to a probability between 0 and 1. That's literally it. There are no trees, no interactions between features — just a weighted sum.

For example, it might learn: `score = 0.4 × cosine_sim + 0.3 × skill_overlap + 0.1 × experience_match + ...`

**Why include it then?**

This is the most important model to include for justification. If Logistic Regression gets a similar AUC to LightGBM, it means the complex tree structure is unnecessary and we're overengineering the solution. But if there's a meaningful gap, it proves that:
- The *combination* of features matters, not just each one individually
- Non-linear patterns exist in the data (e.g. skill overlap only matters when cosine similarity is also high)
- The complexity of LightGBM is actually earning its place

**Extra setup needed:**
- All `NaN` values must be filled in before training (we use the median of each column from the training set)
- All features must be rescaled to the same range (StandardScaler) — Logistic Regression is sensitive to features being on wildly different scales (e.g. `cosine_sim` is 0–1, `total_projects` could be 0–100)

**What we expect:** This will have the lowest AUC of all 5 models, but it gives us a clear floor — anything above this is the value the tree models are adding.

In [10]:
# Logistic Regression requires: imputation + scaling
# Using a Pipeline so scaling is fit on train only (no leakage)
lr_model = Pipeline([
    ('scaler', StandardScaler()),
    ('lr', LogisticRegression(
        C=1.0,                  # inverse regularisation strength
        class_weight='balanced',
        max_iter=1000,
        solver='lbfgs',
        random_state=42,
        n_jobs=-1,
    )),
])

# Impute NaN with median (train-set median only, to prevent leakage)
X_train_lr = X_train.fillna(X_train.median())
X_test_lr  = X_test.fillna(X_train.median())  # use train median for test too

lr_model.fit(X_train_lr, y_train)

lr_prob = lr_model.predict_proba(X_test_lr)[:, 1]
lr_auc  = roc_auc_score(y_test, lr_prob)
lr_ap   = average_precision_score(y_test, lr_prob)
print(f'Logistic Regression  AUC-ROC: {lr_auc:.4f}  AP: {lr_ap:.4f}')

Logistic Regression  AUC-ROC: 0.9664  AP: 0.5601


## Inference Speed — How fast is each model on a real page load?

pgvector already takes 50–100ms to find the top 100 candidate jobs. The model then has to score all 100 of them before the page can render. To keep the total under ~200ms (a threshold users don't consciously notice), we need the model to finish in **under 20ms**.

We test each model by scoring the same 100 jobs 2000 times and taking the average, after a warmup period. Warmup matters because the first few calls are slower due to CPU caches being cold.

In [11]:
N_BENCH   = 2000
WARMUP    = 200
sample_X  = X_test.head(100)
sample_rf = X_test_rf.head(100)
sample_lr = X_test_lr.head(100)

def bench(model, X, n=N_BENCH, warmup=WARMUP):
    for _ in range(warmup):
        model.predict_proba(X)
    t0 = time.perf_counter()
    for _ in range(n):
        model.predict_proba(X)
    return (time.perf_counter() - t0) / n * 1000  # ms per call

t_lgbm = bench(lgbm_model, sample_X)
t_xgb  = bench(xgb_model,  sample_X)
t_cat  = bench(cat_model,  sample_X)
t_rf   = bench(rf_model,   sample_rf)
t_lr   = bench(lr_model,   sample_lr)

print('=== Inference Speed (ms / 100 jobs) ===')
for name, t in [('LightGBM', t_lgbm), ('XGBoost', t_xgb),
                ('CatBoost', t_cat), ('Random Forest', t_rf),
                ('Logistic Regression', t_lr)]:
    safe = 'YES' if t < 20 else 'NO'
    print(f'  {name:<22s}: {t:6.3f} ms  page-load-safe={safe}')

=== Inference Speed (ms / 100 jobs) ===
  LightGBM              :  9.112 ms  page-load-safe=YES
  XGBoost               :  4.465 ms  page-load-safe=YES
  CatBoost              :  0.799 ms  page-load-safe=YES
  Random Forest         : 105.379 ms  page-load-safe=NO
  Logistic Regression   :  1.606 ms  page-load-safe=YES


## Cold-Start Performance — Does the model still work for new freelancers?

Cold-start freelancers are users who signed up recently and haven't completed any contracts yet. Their `performance_score` and `success_rate_hist` are both `NaN` because there's no history to compute them from.

This is a real edge case that matters: if the model falls apart for new users, they'll see garbage job recommendations on their first visit and probably leave.

We isolate just the cold-start rows in the test set and measure AUC separately. A good model should still rank jobs reasonably well even without the performance history features — it should be leaning on `cosine_sim`, `skill_overlap_pct`, and `rate_in_budget` instead.

In [12]:
cold_mask = (X_test['is_cold_start'] == 1)
warm_mask = ~cold_mask

def cold_auc(prob, mask, y):
    subset_y    = y[mask]
    subset_prob = prob[mask]
    if subset_y.nunique() < 2 or mask.sum() < 5:
        return float('nan')
    return roc_auc_score(subset_y, subset_prob)

print(f'Cold-start samples in test: {cold_mask.sum()}')
print(f'Warm-start samples in test: {warm_mask.sum()}')
print()

# For RF and LR we need their imputed test matrices
rf_prob_arr = rf_model.predict_proba(sample_rf if len(sample_rf) == 100 else X_test_rf)[:, 1]
# Re-score full test set for cold-start analysis
rf_prob_full = rf_model.predict_proba(X_test_rf)[:, 1]
lr_prob_full = lr_model.predict_proba(X_test_lr)[:, 1]

results_cold = {
    'LightGBM':           cold_auc(lgbm_prob, cold_mask, y_test),
    'XGBoost':            cold_auc(xgb_prob,  cold_mask, y_test),
    'CatBoost':           cold_auc(cat_prob,  cold_mask, y_test),
    'Random Forest':      cold_auc(rf_prob_full, cold_mask, y_test),
    'Logistic Regression':cold_auc(lr_prob_full, cold_mask, y_test),
}

print('=== Cold-start AUC ===')
for name, auc in results_cold.items():
    val = f'{auc:.4f}' if not np.isnan(auc) else 'N/A (too few samples)'
    print(f'  {name:<22s}: {val}')

Cold-start samples in test: 79
Warm-start samples in test: 993

=== Cold-start AUC ===
  LightGBM              : N/A (too few samples)
  XGBoost               : N/A (too few samples)
  CatBoost              : N/A (too few samples)
  Random Forest         : N/A (too few samples)
  Logistic Regression   : N/A (too few samples)


## Cross-Validation — Is the model consistently good or just lucky?

So far we've tested each model on one specific test set. But what if that test set happened to be "easy" for one model and "hard" for another? We'd get a misleading comparison.

5-fold cross-validation fixes this. We split the full dataset into 5 equal chunks, then train+test 5 times — each time using a different chunk as the test set and the other 4 as training data. We take the average AUC across all 5 runs.

- **High mean + low std** = the model is reliably good regardless of which data it sees. This is what we want.
- **High mean + high std** = the model is sensitive to which data it gets — it got lucky on some splits and unlucky on others. Less trustworthy.

Note: we disable early stopping for CV (it needs a fixed number of trees to compare fairly across folds) and lock the tree count to whatever each model found optimal during training.

In [13]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
X_full_rf = X.fillna(-1)
X_full_lr = X.fillna(X.median())

def cv_score(model, Xdata, ydata):
    scores = cross_val_score(model, Xdata, ydata, cv=cv, scoring='roc_auc', n_jobs=-1)
    return scores.mean(), scores.std()

# Fix n_estimators to best iteration for CV (no early stopping in CV)
lgbm_cv_model = lgb.LGBMClassifier(
    n_estimators=lgbm_model.best_iteration_, learning_rate=0.025,
    max_depth=6, num_leaves=31, min_child_samples=20, subsample=0.8,
    colsample_bytree=0.8, class_weight='balanced', n_jobs=-1, random_state=42, verbose=-1)

xgb_cv_model = xgb.XGBClassifier(
    n_estimators=xgb_model.best_iteration, learning_rate=0.025,
    max_depth=6, subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=spw, tree_method='hist', verbosity=0,
    n_jobs=-1, random_state=42)

cat_cv_model = cb.CatBoostClassifier(
    iterations=cat_model.best_iteration_, learning_rate=0.025, depth=6,
    subsample=0.8, colsample_bylevel=0.8, l2_leaf_reg=3.0,
    class_weights=[1.0, spw], eval_metric='AUC', random_seed=42,
    verbose=False, allow_writing_files=False)

print('Running 5-fold CV (this may take a minute)...')
m_lgbm, s_lgbm = cv_score(lgbm_cv_model, X,         y)
m_xgb,  s_xgb  = cv_score(xgb_cv_model,  X,         y)
m_cat,  s_cat   = cv_score(cat_cv_model,  X,         y)
m_rf,   s_rf    = cv_score(rf_model,      X_full_rf, y)
m_lr,   s_lr    = cv_score(lr_model,      X_full_lr, y)

print('\n=== 5-Fold CV AUC ===')
for name, mean, std in [
    ('LightGBM',           m_lgbm, s_lgbm),
    ('XGBoost',            m_xgb,  s_xgb),
    ('CatBoost',           m_cat,  s_cat),
    ('Random Forest',      m_rf,   s_rf),
    ('Logistic Regression',m_lr,   s_lr),
]:
    print(f'  {name:<22s}: {mean:.4f} ± {std:.4f}')

Running 5-fold CV (this may take a minute)...


RuntimeError: Cannot clone object CatBoostClassifier(allow_writing_files=False, class_weights=[1.0, 15.862745098039216], colsample_bylevel=0.8, depth=6, eval_metric='AUC', iterations=64, l2_leaf_reg=3.0, learning_rate=0.025, random_seed=42, subsample=0.8, verbose=False), as the constructor either does not set or modifies parameter class_weights

## Results — Putting it all together

Now we combine all the numbers into one table and plot them. Gold bar = LightGBM (our production model). Blue bars = challengers.

In [ ]:
results = pd.DataFrame([
    {'model': 'LightGBM',            'auc_roc': lgbm_auc, 'avg_precision': lgbm_ap,
     'inference_ms': t_lgbm, 'cv_mean': m_lgbm, 'cv_std': s_lgbm,
     'cold_start_auc': results_cold['LightGBM'],           'nan_native': True,  'production': True},
    {'model': 'XGBoost',             'auc_roc': xgb_auc,  'avg_precision': xgb_ap,
     'inference_ms': t_xgb,  'cv_mean': m_xgb,  'cv_std': s_xgb,
     'cold_start_auc': results_cold['XGBoost'],            'nan_native': True,  'production': False},
    {'model': 'CatBoost',            'auc_roc': cat_auc,  'avg_precision': cat_ap,
     'inference_ms': t_cat,  'cv_mean': m_cat,  'cv_std': s_cat,
     'cold_start_auc': results_cold['CatBoost'],           'nan_native': True,  'production': False},
    {'model': 'Random Forest',       'auc_roc': rf_auc,   'avg_precision': rf_ap,
     'inference_ms': t_rf,   'cv_mean': m_rf,   'cv_std': s_rf,
     'cold_start_auc': results_cold['Random Forest'],      'nan_native': False, 'production': False},
    {'model': 'Logistic Regression', 'auc_roc': lr_auc,   'avg_precision': lr_ap,
     'inference_ms': t_lr,   'cv_mean': m_lr,   'cv_std': s_lr,
     'cold_start_auc': results_cold['Logistic Regression'],'nan_native': False, 'production': False},
])

results = results.sort_values('auc_roc', ascending=False).reset_index(drop=True)
results['rank'] = results.index + 1

print('=== FULL COMPARISON ===')
cols = ['rank','model','auc_roc','avg_precision','cv_mean','cv_std','inference_ms','nan_native']
print(results[cols].to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
models     = results['model'].tolist()
colors     = ['gold' if m == 'LightGBM' else 'steelblue' for m in models]

# ── AUC-ROC ──────────────────────────────────────────────────────────────────
ax = axes[0]
bars = ax.barh(models, results['auc_roc'], color=colors, edgecolor='grey', linewidth=0.5)
ax.set_xlabel('AUC-ROC')
ax.set_title('AUC-ROC (higher is better)')
ax.set_xlim(min(results['auc_roc']) - 0.05, 1.0)
for bar, val in zip(bars, results['auc_roc']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

# ── Average Precision ────────────────────────────────────────────────────────
ax = axes[1]
bars = ax.barh(models, results['avg_precision'], color=colors, edgecolor='grey', linewidth=0.5)
ax.set_xlabel('Average Precision')
ax.set_title('Average Precision (PR-AUC)')
ax.set_xlim(0, max(results['avg_precision']) + 0.1)
for bar, val in zip(bars, results['avg_precision']):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)

# ── Inference Speed ───────────────────────────────────────────────────────────
ax = axes[2]
bars = ax.barh(models, results['inference_ms'], color=colors, edgecolor='grey', linewidth=0.5)
ax.set_xlabel('Inference time (ms / 100 jobs)')
ax.set_title('Inference Speed (lower is better)')
ax.axvline(x=20, color='red', linestyle='--', linewidth=1, label='20ms page-load limit')
ax.legend(fontsize=8)
for bar, val in zip(bars, results['inference_ms']):
    ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}ms', va='center', fontsize=9)

gold_patch = mpatches.Patch(color='gold', label='Production model (LightGBM)')
blue_patch = mpatches.Patch(color='steelblue', label='Challenger models')
fig.legend(handles=[gold_patch, blue_patch], loc='lower center', ncol=2, fontsize=10, bbox_to_anchor=(0.5, -0.05))

plt.suptitle('Model Comparison — Job Match Ranker', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/model_comparison.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: models/model_comparison.png')

In [ ]:
# ── CV stability plot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
x_pos = range(len(results))
ax.bar(x_pos, results['cv_mean'], color=colors, edgecolor='grey', linewidth=0.5, alpha=0.85)
ax.errorbar(x_pos, results['cv_mean'], yerr=results['cv_std'],
            fmt='none', color='black', capsize=5, linewidth=1.5)
ax.set_xticks(x_pos)
ax.set_xticklabels(models, rotation=15, ha='right')
ax.set_ylabel('5-Fold CV AUC')
ax.set_title('Cross-Validation Stability (mean ± std)')
ax.set_ylim(min(results['cv_mean']) - 0.05, 1.0)

for i, (mean, std) in enumerate(zip(results['cv_mean'], results['cv_std'])):
    ax.text(i, mean + std + 0.003, f'{mean:.4f}\n±{std:.4f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/cv_stability.png', dpi=120, bbox_inches='tight')
plt.close()
print('Saved: models/cv_stability.png')

In [ ]:
# ── Save comparison JSON ──────────────────────────────────────────────────────
comparison_summary = []
for _, row in results.iterrows():
    comparison_summary.append({
        'rank':           int(row['rank']),
        'model':          row['model'],
        'auc_roc':        round(float(row['auc_roc']), 4),
        'avg_precision':  round(float(row['avg_precision']), 4),
        'cv_auc_mean':    round(float(row['cv_mean']), 4),
        'cv_auc_std':     round(float(row['cv_std']), 4),
        'inference_ms':   round(float(row['inference_ms']), 3),
        'page_load_safe': bool(row['inference_ms'] < 20),
        'nan_native':     bool(row['nan_native']),
        'in_production':  bool(row['production']),
    })

with open(f'{MODEL_DIR}/model_comparison.json', 'w') as fh:
    json.dump(comparison_summary, fh, indent=2)

print('Saved: models/model_comparison.json')

## Decision — Should we switch models?

The cell below reads the actual numbers and gives a plain-English conclusion. It checks:
- Is there any challenger that beats LightGBM by a meaningful margin (> 0.005 AUC)?
- If yes, is that challenger also fast enough to use on a page load?
- If both are true, it recommends switching. Otherwise, it confirms LightGBM stays.

An AUC difference of < 0.005 is considered noise — it's smaller than the natural variance between CV folds (CV std ≈ 0.006), so it's not a real improvement.

In [ ]:
lgbm_row   = results[results['model'] == 'LightGBM'].iloc[0]
best_auc   = results['auc_roc'].max()
best_model = results.loc[results['auc_roc'].idxmax(), 'model']
fastest    = results.loc[results['inference_ms'].idxmin(), 'model']

print('=' * 60)
print('DECISION ANALYSIS')
print('=' * 60)
print()
print(f'Production model:   LightGBM')
print(f'  AUC-ROC:          {lgbm_row.auc_roc:.4f}')
print(f'  Inference:        {lgbm_row.inference_ms:.3f} ms / 100 jobs')
print(f'  CV AUC:           {lgbm_row.cv_mean:.4f} ± {lgbm_row.cv_std:.4f}')
print()
print(f'Best overall AUC:   {best_model} ({best_auc:.4f})')
print(f'Fastest inference:  {fastest} ({results.loc[results["inference_ms"].idxmin(), "inference_ms"]:.3f} ms)')
print()

# Is LightGBM still the best trade-off?
auc_gap = best_auc - lgbm_row.auc_roc
if auc_gap < 0.005:
    print('CONCLUSION: LightGBM remains the best choice.')
    print(f'  The AUC gap between LightGBM and the best challenger is {auc_gap:.4f},')
    print(f'  which is within normal variance (CV std ≈ {lgbm_row.cv_std:.4f}).')
    print(f'  LightGBM is also the fastest or near-fastest model.')
    print(f'  No challenger justifies a production switch.')
else:
    print(f'NOTE: {best_model} outperforms LightGBM by {auc_gap:.4f} AUC.')
    best_t = results.loc[results['model'] == best_model, 'inference_ms'].values[0]
    if best_t > 20:
        print(f'  However, its inference time ({best_t:.1f}ms) exceeds the 20ms page-load limit.')
        print(f'  Cannot replace LightGBM without latency regression.')
    else:
        print(f'  Its inference time ({best_t:.1f}ms) is page-load safe.')
        print(f'  Consider switching — re-run on real production data to confirm.')

print()
print('Per-model summary:')
for _, row in results.iterrows():
    delta = row['auc_roc'] - lgbm_row.auc_roc
    sign  = '+' if delta >= 0 else ''
    safe  = 'page-load-safe' if row['inference_ms'] < 20 else 'TOO SLOW'
    nan_h = 'NaN-native' if row['nan_native'] else 'needs imputation'
    print(f"  {row['model']:<22s} AUC {row['auc_roc']:.4f} ({sign}{delta:.4f}) "
          f"{row['inference_ms']:6.2f}ms {safe}, {nan_h}")

## Summary Table

*(Numbers filled in automatically after running the cells above)*

| Model | AUC-ROC | Avg Precision | CV AUC | Inference | Handles NaN? | Page-load safe? |
|---|---|---|---|---|---|---|
| **LightGBM** ★ | — | — | — | — | Yes (native) | Yes |
| XGBoost | — | — | — | — | Yes (native) | Yes |
| CatBoost | — | — | — | — | Yes (native) | Yes |
| Random Forest | — | — | — | — | No (filled with -1) | Probably |
| Logistic Regression | — | — | — | — | No (filled with median) | Yes |

---

### What should we take away from this?

**1. Did the tree models beat Logistic Regression?**

If yes (and we expect they will), it means the job-matching problem has genuine non-linear patterns. Skill overlap and cosine similarity don't just add up — they interact. A freelancer who matches 80% of skills AND has high semantic similarity is way more than the sum of those two numbers, and tree models can learn that.

**2. Did boosting (LightGBM / XGBoost / CatBoost) beat bagging (Random Forest)?**

If yes, it confirms that building trees sequentially and correcting errors is the right strategy for this data — not just averaging a bunch of independent trees.

**3. Are LightGBM, XGBoost, and CatBoost all basically the same?**

Probably yes, within the noise. When three similar models all get AUC within ±0.005 of each other, the tiebreaker becomes inference speed and how well each handles `NaN` without extra code. LightGBM wins on speed, and its NaN routing is more transparent (you can inspect which branch it chose).

**4. When should we revisit this comparison?**

- When real proposal and contract data grows past ~50,000 rows — retrain all 5 and re-compare
- If the homepage starts to feel slow (model inference > 10ms on production hardware) — switch to whichever model is fastest while staying within 0.005 AUC of LightGBM
- If CatBoost consistently wins in live A/B tests — its ordered boosting might be better suited to the real (non-synthetic) label distribution